In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/activity_labels.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/README.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features_info.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/subject_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/X_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_gyr

# Loading and Normalizing the dataset

In [2]:
import numpy as np

def load_raw_signals(folder_path, filenames):
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt"
        data = np.loadtxt(file_path)
        signals.append(data)
    
    return np.transpose(np.array(signals), (1, 2, 0))
filenames = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z'
]
filenames_train = [f + '_train' for f in filenames]
filenames_test = [f + '_test' for f in filenames]
raw_data_path_train = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals'
raw_data_path_test =  '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals'

x_train_raw = load_raw_signals(raw_data_path_train, filenames_train)
X_test_raw = load_raw_signals(raw_data_path_test, filenames_test)


mean = np.mean(x_train_raw, axis=(0, 1))
std = np.std(x_train_raw, axis=(0, 1))

#normalizing 
x_train = (x_train_raw - mean) / (std + 1e-8)
X_test = (X_test_raw - mean) / (std + 1e-8) 

# loading labels
y_train = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/y_train.txt', dtype=int) - 1
y_test = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt', dtype=int) - 1

print(f"Normalized x_train shape: {x_train.shape}")
print(f"Normalized X_test shape: {X_test.shape}")

Normalized x_train shape: (7352, 128, 6)
Normalized X_test shape: (2947, 128, 6)


# Coding the 1D-CNN Architecture

In [3]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, models
model = keras.Sequential(name="HAR_1DCNN") 
model.add(layers.Input(shape=(128, 6))) # loading an empty stack to which we'll add layers
model.add(layers.Conv1D(   #Now we're adding layers to this 
    filters=64,    #this arg defines how many local features we'll look for
    kernel_size=5, #this is essentially our window size which we slide over our 128 timesteps to extract features
    activation='relu', #rectified linear unit introduces non-linearity basically acting as a gate to decide which information is important enough to be passed to the next layer
    padding= 'same',   #This makes sure that our output is the same length as our input adds zeros so the kernel can center itself
    #input_shape=(128,6) #This defines the size of our input passed 
))
#Stacking another layer cause we want a deeper model 
model.add(layers.MaxPooling1D(pool_size=2)) 
model.add(layers.Dropout(rate=0.5)) 

model.add(layers.Conv1D(   
    filters=128,   
    kernel_size=3,
    activation='relu', 
    padding= 'same',  
))
model.add(layers.MaxPooling1D(pool_size=2)) # makes model invincible to temporal shifts (like a pattern that occurs at timestep 11 and 12 are for the same motion provided the pattern is similar)
model.add(layers.Dropout(rate=0.5)) # turns off 50% of neurons mid training to make sure the model doesnt memorize 
model.add(layers.Flatten()) # converts our 2D matrix (timesteps x features) into one long 1D vector


model.add(layers.Dense(128, activation='relu')) #input neurons


model.add(layers.Dropout(0.5)) #turns off 50% of neurons mid training to prevent memorization


model.add(layers.Dense(6, activation='softmax')) #softmax is a function that calculates probabilities of each of 6 outcomes and chooses the highest probability option
model.compile(
    optimizer='adam',  # updates weights and bias
    loss='sparse_categorical_crossentropy', #our loss function
    metrics=['accuracy'] # what metrics we'll be seeing
)

model.summary() # summary

history = model.fit(    #actual training logic epochs means training cycles btw
    x_train, y_train,
    epochs=30, # this just means the model will run 30 epochs
    batch_size=16, # the model will look at 16 samples at ones
    validation_data=(X_test, y_test), #validates accuracy against test data
    shuffle=True,   # shuffling the data
)

model.save('my_model.h5')

2026-05-14 17:16:43.340401: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778779003.750795      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778779003.876811      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778779004.920169      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778779004.920229      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778779004.920233      24 computation_placer.cc:177] computation placer alr

Model: "HAR_1DCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 128, 64)        │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 551,878 (2.11 MB)

 Trainable params: 551,878 (2.11 MB)

...

Epoch 1/30


I0000 00:00:1778779051.940976      72 service.cc:152] XLA service 0x7841c800c110 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778779051.941009      72 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778779051.941013      72 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778779052.391801      72 cuda_dnn.cc:529] Loaded cuDNN version 91002


 57/460 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2018 - loss: 1.9242

I0000 00:00:1778779056.388835      72 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


460/460 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.3409 - loss: 1.3940 - val_accuracy: 0.6244 - val_loss: 0.8247
Epoch 2/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6228 - loss: 0.7389 - val_accuracy: 0.7021 - val_loss: 0.6375
Epoch 3/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7104 - loss: 0.5839 - val_accuracy: 0.7231 - val_loss: 0.6078
Epoch 4/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7810 - loss: 0.4807 - val_accuracy: 0.7608 - val_loss: 0.5749
Epoch 5/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7925 - loss: 0.4498 - val_accuracy: 0.7777 - val_loss: 0.5323
Epoch 6/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8245 - loss: 0.4107 - val_accuracy: 0.8490 - val_loss: 0.4149
Epoch 7/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8340 - loss: 0.3732 - val_accuracy: 0.8751 - val_loss: 0.3685
Epoch 8/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8718 - loss: 0.3295 - val_accuracy: 0.8877 - va

# Converting the .h5 model to .tflite model

In [4]:
import tensorflow as tf
model=tf.keras.models.load_model('my_model.h5') # loads the saved model

converter=tf.lite.TFLiteConverter.from_keras_model(model) # initializes the converter
tflite_fp32model=converter.convert() # does the converting here 
with open('model_fp32.tflite','wb') as f: # writing the new tflite model to a file named 'model_fp32.tflite' btw wb means write binary
    f.write(tflite_fp32model)

INFO:tensorflow:Assets written to: /tmp/tmpmn0ntfpg/assets


INFO:tensorflow:Assets written to: /tmp/tmpmn0ntfpg/assets


Saved artifact at '/tmp/tmpmn0ntfpg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  132227544627920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227544627536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227544624656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227544627152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227544622928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227544625808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227544626384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227544622736: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1778779113.742209      24 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778779113.742251      24 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1778779113.748705      24 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
